In [ ]:
from pathlib import Path
import openslide
import numpy as np
import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle


def select_level_tiles_from_coarse_thumbnail_contour(
    slide_path,
    target_level=4,
    tile_w=800,
    tile_h=1400,

    # --- nueva detección del contorno en thumbnail ---
    thumb_sat_threshold=15,
    thumb_val_threshold=220,
    thumb_dist_threshold=20,
    thumb_kernel_open_size=5,
    thumb_kernel_close_size=9,
    thumb_kernel_smooth_size=7,
    contour_band_thickness_thumb=10,

    # --- filtro de tiles en level ---
    specimen_frac_threshold=0.02,
    dist_threshold=18,
    sat_threshold=18,
    value_threshold=220,
    min_contour_overlap_pixels=5,
    show_removed=True,
):
    """
    Pipeline:
    1) Detecta contorno aproximado del espécimen en thumbnail usando:
       - HSV
       - distancia al color de fondo estimado con bordes
       - componente conectado mayor
       - relleno de huecos
       - suavizado morfológico
    2) Genera una banda de contorno en thumbnail
    3) Segmenta el espécimen en target_level para saber qué tiles contienen tejido
    4) Construye tiles en target_level
    5) Conserva solo tiles que:
       - no son 100% negros
       - contienen espécimen suficiente
       - intersectan la banda de contorno detectada en thumbnail
    6) Muestra:
       - contorno final sobre thumbnail
       - todas las cuadrículas
       - cuadrículas seleccionadas por contorno
       - tiles seleccionados uno por uno en level4
    """

    # =========================================================
    # 0. Abrir slide
    # =========================================================
    slide_path = Path(slide_path)
    slide = openslide.OpenSlide(str(slide_path))

    print("Level dimensions:", slide.level_dimensions)
    print("Level downsamples:", slide.level_downsamples)

    lvl_w, lvl_h = slide.level_dimensions[target_level]
    print(f"\nAnalizando level {target_level}: {lvl_w} x {lvl_h} px")

    # =========================================================
    # 1. Leer thumbnail
    # =========================================================
    thumb = slide.associated_images["thumbnail"].convert("RGB")
    thumb_np = np.array(thumb)
    thumb_h, thumb_w = thumb_np.shape[:2]

    # =========================================================
    # 2. NUEVA detección de contorno en thumbnail
    #    HSV + distancia al fondo + componente mayor + suavizado
    # =========================================================
    img = thumb_np.copy()

    hsv_thumb = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    _, S_thumb, V_thumb = cv2.split(hsv_thumb)

    # estimar color de fondo con los bordes de la thumbnail
    border_pixels = np.concatenate([
        img[0, :, :],
        img[-1, :, :],
        img[:, 0, :],
        img[:, -1, :]
    ], axis=0).astype(np.float32)

    bg_color = np.median(border_pixels, axis=0)

    # distancia de cada píxel al color de fondo
    dist_thumb = np.linalg.norm(
        img.astype(np.float32) - bg_color[None, None, :],
        axis=2
    )

    # máscaras básicas
    mask_hsv = ((S_thumb > thumb_sat_threshold) | (V_thumb < thumb_val_threshold)).astype(np.uint8) * 255
    mask_dist = (dist_thumb > thumb_dist_threshold).astype(np.uint8) * 255

    # combinar HSV + distancia
    mask_tissue_thumb = np.maximum(mask_hsv, mask_dist)

    # limpieza morfológica
    kernel_open = np.ones((thumb_kernel_open_size, thumb_kernel_open_size), np.uint8)
    kernel_close = np.ones((thumb_kernel_close_size, thumb_kernel_close_size), np.uint8)

    mask_tissue_thumb = cv2.morphologyEx(mask_tissue_thumb, cv2.MORPH_OPEN, kernel_open)
    mask_tissue_thumb = cv2.morphologyEx(mask_tissue_thumb, cv2.MORPH_CLOSE, kernel_close)

    # quedarse con el mayor componente conectado
    nlab, labels, stats, _ = cv2.connectedComponentsWithStats(mask_tissue_thumb, connectivity=8)

    if nlab <= 1:
        raise ValueError("No se ha detectado espécimen en la thumbnail.")

    largest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    mask_thumb_main = np.uint8(labels == largest) * 255

    # rellenar huecos internos
    h_t, w_t = mask_thumb_main.shape
    flood = mask_thumb_main.copy()
    flood_mask = np.zeros((h_t + 2, w_t + 2), np.uint8)

    cv2.floodFill(flood, flood_mask, (0, 0), 255)
    holes = cv2.bitwise_not(flood)
    mask_thumb_filled = cv2.bitwise_or(mask_thumb_main, holes)

    # suavizar un poco el borde
    kernel_smooth = np.ones((thumb_kernel_smooth_size, thumb_kernel_smooth_size), np.uint8)
    mask_thumb_smooth = cv2.morphologyEx(mask_thumb_filled, cv2.MORPH_CLOSE, kernel_smooth)
    mask_thumb_smooth = cv2.morphologyEx(mask_thumb_smooth, cv2.MORPH_OPEN, kernel_smooth)

    # sacar contorno externo final
    contours_thumb, _ = cv2.findContours(
        mask_thumb_smooth,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_NONE
    )

    if len(contours_thumb) == 0:
        raise ValueError("No se ha encontrado contorno externo en thumbnail.")

    coarse_contour_thumb = max(contours_thumb, key=cv2.contourArea)

    # banda alrededor del contorno para seleccionar tiles
    contour_band_thumb = np.zeros_like(mask_thumb_smooth)
    cv2.drawContours(
        contour_band_thumb,
        [coarse_contour_thumb],
        -1,
        255,
        thickness=contour_band_thickness_thumb
    )

    # Usamos la máscara suavizada como máscara final de thumbnail
    mask_thumb_filled = mask_thumb_smooth

    # =========================================================
    # 3. Mostrar solo el contorno final en thumbnail
    # =========================================================
    overlay_thumb_contour = thumb_np.copy()
    cv2.drawContours(
        overlay_thumb_contour,
        [coarse_contour_thumb],
        -1,
        (255, 255, 0),
        2
    )

    plt.figure(figsize=(8, 14))
    plt.imshow(overlay_thumb_contour)
    plt.axis("off")
    plt.title("Contorno final detectado sobre thumbnail")
    plt.show()

    # =========================================================
    # 4. Leer imagen completa del target_level y segmentar espécimen
    # =========================================================
    img_level = slide.read_region((0, 0), target_level, (lvl_w, lvl_h)).convert("RGB")
    img_level = np.array(img_level)

    img = img_level.astype(np.float32)

    border_pixels = np.concatenate([
        img[0, :, :],
        img[-1, :, :],
        img[:, 0, :],
        img[:, -1, :]
    ], axis=0)

    bg_color = np.median(border_pixels, axis=0)
    dist = np.linalg.norm(img - bg_color[None, None, :], axis=2)

    hsv_level = cv2.cvtColor(img_level, cv2.COLOR_RGB2HSV)
    _, S, V = cv2.split(hsv_level)

    mask = ((dist > dist_threshold) | (S > sat_threshold) | (V < value_threshold)).astype(np.uint8) * 255

    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    nlab, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)

    if nlab <= 1:
        raise ValueError("No se ha detectado espécimen en el target_level.")

    largest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    mask_main = np.uint8(labels == largest) * 255

    # rellenar huecos internos
    h, w = mask_main.shape
    flood = mask_main.copy()
    flood_mask = np.zeros((h + 2, w + 2), np.uint8)
    cv2.floodFill(flood, flood_mask, (0, 0), 255)
    holes = cv2.bitwise_not(flood)
    mask_filled = cv2.bitwise_or(mask_main, holes)

    # =========================================================
    # 5. Construir tiles en target_level
    # =========================================================
    tiles = []
    tile_id = 1

    scale_x = thumb_w / lvl_w
    scale_y = thumb_h / lvl_h

    row_id = 0

    for y0 in range(0, lvl_h, tile_h):
        col_id = 0

        for x0 in range(0, lvl_w, tile_w):
            w_tile = min(tile_w, lvl_w - x0)
            h_tile = min(tile_h, lvl_h - y0)

            tile_img = img_level[y0:y0 + h_tile, x0:x0 + w_tile]
            tile_mask = mask_filled[y0:y0 + h_tile, x0:x0 + w_tile]

            # 100% negro
            is_black = np.all(tile_img == 0)

            # espécimen dentro del tile
            specimen_frac = tile_mask.mean() / 255.0

            # proyección del tile sobre thumbnail
            x0_t = int(np.floor(x0 * scale_x))
            y0_t = int(np.floor(y0 * scale_y))
            x1_t = int(np.ceil((x0 + w_tile) * scale_x))
            y1_t = int(np.ceil((y0 + h_tile) * scale_y))

            x0_t = max(0, min(x0_t, thumb_w))
            x1_t = max(0, min(x1_t, thumb_w))
            y0_t = max(0, min(y0_t, thumb_h))
            y1_t = max(0, min(y1_t, thumb_h))

            if x1_t <= x0_t or y1_t <= y0_t:
                contour_overlap_pixels = 0
            else:
                contour_overlap_pixels = int(
                    np.count_nonzero(contour_band_thumb[y0_t:y1_t, x0_t:x1_t])
                )

            keep = (
                (not is_black)
                and (specimen_frac > specimen_frac_threshold)
                and (contour_overlap_pixels >= min_contour_overlap_pixels)
            )

            reason = []

            if is_black:
                reason.append("100% negra")

            if specimen_frac <= specimen_frac_threshold:
                reason.append(f"sin espécimen suficiente ({specimen_frac:.3f})")

            if contour_overlap_pixels < min_contour_overlap_pixels:
                reason.append(f"sin solape suficiente con contorno ({contour_overlap_pixels})")

            reason_text = "conservar" if keep else ", ".join(reason)

            tiles.append({
                "id": tile_id,
                "row": row_id,
                "col": col_id,
                "x0": x0,
                "y0": y0,
                "w": w_tile,
                "h": h_tile,
                "img": tile_img,
                "is_black": is_black,
                "specimen_frac": specimen_frac,
                "contour_overlap_pixels": contour_overlap_pixels,
                "keep": keep,
                "reason": reason_text,
            })

            tile_id += 1
            col_id += 1

        row_id += 1

    kept_tiles = [t for t in tiles if t["keep"]]
    removed_tiles = [t for t in tiles if not t["keep"]]

    # =========================================================
    # 6. Overview: todas las cuadrículas
    # =========================================================
    fig, ax = plt.subplots(figsize=(8, 14))
    ax.imshow(thumb_np)

    # dibujar contorno aproximado
    cv2_contour = coarse_contour_thumb[:, 0, :]
    ax.plot(cv2_contour[:, 0], cv2_contour[:, 1], color="yellow", linewidth=1.5)

    for t in tiles:
        x_t = t["x0"] * scale_x
        y_t = t["y0"] * scale_y
        w_t = t["w"] * scale_x
        h_t = t["h"] * scale_y

        rect = Rectangle(
            (x_t, y_t),
            w_t,
            h_t,
            fill=False,
            edgecolor="cyan",
            linewidth=0.8
        )
        ax.add_patch(rect)

        ax.text(
            x_t + 2,
            y_t + 12,
            str(t["id"]),
            color="yellow",
            fontsize=7,
            bbox=dict(facecolor="black", alpha=0.6, edgecolor="none", pad=1)
        )

    ax.set_title(f"Todas las cuadrículas - level {target_level} + contorno final")
    ax.axis("off")
    plt.show()

    # =========================================================
    # 7. Overview final: seleccionadas por el contorno
    # =========================================================
    fig, ax = plt.subplots(figsize=(8, 14))
    ax.imshow(thumb_np)

    # banda del contorno
    ax.contour(
        contour_band_thumb > 0,
        levels=[0.5],
        colors="yellow",
        linewidths=1
    )

    for t in tiles:
        x_t = t["x0"] * scale_x
        y_t = t["y0"] * scale_y
        w_t = t["w"] * scale_x
        h_t = t["h"] * scale_y

        if not t["keep"]:
            if not show_removed:
                continue

            color = "red"
            linestyle = "--"
        else:
            color = "lime"
            linestyle = "-"

        rect = Rectangle(
            (x_t, y_t),
            w_t,
            h_t,
            fill=False,
            edgecolor=color,
            linewidth=1.2,
            linestyle=linestyle
        )
        ax.add_patch(rect)

        ax.text(
            x_t + 2,
            y_t + 12,
            str(t["id"]),
            color=color,
            fontsize=7,
            bbox=dict(facecolor="black", alpha=0.6, edgecolor="none", pad=1)
        )

    ax.set_title(
        f"Tiles seleccionados por contorno final - level {target_level}\n"
        f"Verde = conservar | Rojo = eliminar"
    )
    ax.axis("off")
    plt.show()

    # =========================================================
    # 8. Texto
    # =========================================================
    print("\n==============================")
    print("TILES ELIMINADOS")
    print("==============================")

    if len(removed_tiles) == 0:
        print("Ninguno")
    else:
        for t in removed_tiles:
            print(
                f"Tile {t['id']:02d} | "
                f"row={t['row']}, col={t['col']} | "
                f"x={t['x0']}, y={t['y0']}, w={t['w']}, h={t['h']} | "
                f"motivo: {t['reason']}"
            )

    print("\n==============================")
    print("TILES CONSERVADOS POR CONTORNO")
    print("==============================")

    if len(kept_tiles) == 0:
        print("Ninguno")
    else:
        for t in kept_tiles:
            print(
                f"Tile {t['id']:02d} | "
                f"row={t['row']}, col={t['col']} | "
                f"x={t['x0']}, y={t['y0']}, w={t['w']}, h={t['h']} | "
                f"specimen_frac={t['specimen_frac']:.3f} | "
                f"contour_overlap_pixels={t['contour_overlap_pixels']}"
            )

    # =========================================================
    # 9. Mostrar tiles conservados uno por uno en target_level
    # =========================================================
    print("\n==============================")
    print("MOSTRANDO TILES CONSERVADOS")
    print("==============================")

    for t in kept_tiles:
        h_img, w_img = t["img"].shape[:2]
        aspect = h_img / w_img

        fig_w = 10
        fig_h = min(16, max(6, fig_w * aspect))

        plt.figure(figsize=(fig_w, fig_h))
        plt.imshow(t["img"])
        plt.axis("off")
        plt.title(
            f"Tile {t['id']} - level {target_level}\n"
            f"row={t['row']}, col={t['col']} | "
            f"x={t['x0']}, y={t['y0']}, w={t['w']}, h={t['h']} | "
            f"specimen_frac={t['specimen_frac']:.3f} | "
            f"contour_overlap_pixels={t['contour_overlap_pixels']}"
        )
        plt.show()

    return (
        kept_tiles,
        removed_tiles,
        mask_thumb_filled,
        coarse_contour_thumb,
        contour_band_thumb,
        mask_filled
    )

In [ ]:
slide_path = r"Datos\SB013\SB013\H&E_python\SB013.mrxs"

kept_tiles, removed_tiles, mask_thumb_filled, coarse_contour_thumb, contour_band_thumb, mask_filled = \
    select_level_tiles_from_coarse_thumbnail_contour(
        slide_path=slide_path,
        target_level=4,
        tile_w=800,
        tile_h=1400,
        specimen_frac_threshold=0.02,
        contour_band_thickness_thumb=10,
        min_contour_overlap_pixels=5
    )

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt


def contorno_abierto_superior_he(
    img_rgb,
    sat_threshold=20,
    min_value=35,
    kernel_open_size=3,
    kernel_close_size=15,
    min_component_area=500,
    smooth_window=21,
    dot_spacing=6,
    dot_radius=3,
    dot_color=(255, 255, 0),
    show_steps=True
):
    """
    Extrae SOLO la frontera superior entre background y espécimen
    como una línea ABIERTA (no cerrada), ignorando laterales.

    Pensado para imágenes H&E donde:
    - fondo = blanco
    - puede haber zonas negras fuera de la imagen útil
    - espécimen = rosa/morado
    - hay huecos/manchas blancas dentro del espécimen
    """

    img = img_rgb.copy()
    h, w = img.shape[:2]

    # =========================================================
    # 1. Máscara de tejido
    #    Detectamos tejido teñido y evitamos negro
    # =========================================================
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    H, S, V = cv2.split(hsv)

    # tejido: saturación suficiente y no demasiado oscuro
    mask = ((S > sat_threshold) & (V > min_value)).astype(np.uint8) * 255

    # =========================================================
    # 2. Limpieza morfológica
    # =========================================================
    kernel_open = np.ones((kernel_open_size, kernel_open_size), np.uint8)
    kernel_close = np.ones((kernel_close_size, kernel_close_size), np.uint8)

    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel_open)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel_close)

    # =========================================================
    # 3. Eliminar componentes pequeños (ruido)
    # =========================================================
    nlab, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)

    mask_clean = np.zeros_like(mask)
    for lab in range(1, nlab):
        area = stats[lab, cv2.CC_STAT_AREA]
        if area >= min_component_area:
            mask_clean[labels == lab] = 255

    # =========================================================
    # 4. Sacar SOLO la frontera superior:
    #    para cada columna x, primer píxel de tejido desde arriba
    # =========================================================
    xs = []
    ys = []

    for x in range(w):
        ys_col = np.where(mask_clean[:, x] > 0)[0]
        if len(ys_col) > 0:
            xs.append(x)
            ys.append(int(ys_col[0]))

    if len(xs) == 0:
        raise ValueError("No se ha encontrado frontera tejido-background.")

    xs = np.array(xs)
    ys = np.array(ys, dtype=np.float32)

    # =========================================================
    # 5. Suavizar la curva para que no quede dentada
    # =========================================================
    if smooth_window > 1:
        if smooth_window % 2 == 0:
            smooth_window += 1

        pad = smooth_window // 2
        ys_pad = np.pad(ys, (pad, pad), mode="edge")
        kernel = np.ones(smooth_window, dtype=np.float32) / smooth_window
        ys_smooth = np.convolve(ys_pad, kernel, mode="valid")
    else:
        ys_smooth = ys.copy()

    ys_smooth = np.round(ys_smooth).astype(int)

    # =========================================================
    # 6. Dibujar la línea abierta como línea de puntos gruesa
    # =========================================================
    overlay = img.copy()

    for i in range(0, len(xs), dot_spacing):
        cv2.circle(
            overlay,
            (int(xs[i]), int(ys_smooth[i])),
            dot_radius,
            dot_color,
            thickness=-1
        )

    # También generamos una versión continua más gruesa
    # por si quieres verla aún más clara
    overlay_line = img.copy()
    pts = np.stack([xs, ys_smooth], axis=1).reshape(-1, 1, 2).astype(np.int32)
    cv2.polylines(
        overlay_line,
        [pts],
        isClosed=False,
        color=dot_color,
        thickness=max(2, dot_radius)
    )

    if show_steps:
        fig, axes = plt.subplots(1, 4, figsize=(20, 7))

        axes[0].imshow(img)
        axes[0].axis("off")
        axes[0].set_title("Imagen original")

        axes[1].imshow(mask_clean, cmap="gray")
        axes[1].axis("off")
        axes[1].set_title("Máscara de tejido")

        axes[2].imshow(overlay)
        axes[2].axis("off")
        axes[2].set_title("Frontera abierta punteada")

        axes[3].imshow(overlay_line)
        axes[3].axis("off")
        axes[3].set_title("Frontera abierta gruesa")

        plt.tight_layout()
        plt.show()

    return overlay, overlay_line, mask_clean, xs, ys_smooth

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt


def fill_holes_binary(mask):
    """
    Rellena huecos internos de una máscara binaria.
    mask debe ser uint8 con valores 0/255.
    """
    mask = mask.copy()

    padded = cv2.copyMakeBorder(
        mask,
        1, 1, 1, 1,
        borderType=cv2.BORDER_CONSTANT,
        value=0
    )

    flood = padded.copy()
    flood_mask = np.zeros((padded.shape[0] + 2, padded.shape[1] + 2), np.uint8)

    cv2.floodFill(flood, flood_mask, (0, 0), 255)

    holes = cv2.bitwise_not(flood)
    filled = cv2.bitwise_or(padded, holes)

    return filled[1:-1, 1:-1]


def keep_border_connected_components(mask, min_area=500):
    """
    Se queda solo con componentes blancos conectados al borde de la imagen.
    Esto permite identificar fondo blanco exterior y descartar huecos internos.
    """
    h, w = mask.shape[:2]

    nlab, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)

    out = np.zeros_like(mask)

    for lab in range(1, nlab):
        x = stats[lab, cv2.CC_STAT_LEFT]
        y = stats[lab, cv2.CC_STAT_TOP]
        bw = stats[lab, cv2.CC_STAT_WIDTH]
        bh = stats[lab, cv2.CC_STAT_HEIGHT]
        area = stats[lab, cv2.CC_STAT_AREA]

        touches_border = (
            x == 0 or
            y == 0 or
            x + bw >= w or
            y + bh >= h
        )

        if touches_border and area >= min_area:
            out[labels == lab] = 255

    return out


def get_open_runs_from_closed_contour(points, keep_bool, min_len=30):
    """
    Dado un contorno cerrado y una máscara booleana de puntos válidos,
    devuelve segmentos abiertos consecutivos.
    """
    points = np.asarray(points)
    keep_bool = np.asarray(keep_bool).astype(bool)

    n = len(points)
    if n == 0 or not np.any(keep_bool):
        return []

    # Si el run cruza el inicio/final del contorno cerrado, rotamos
    if keep_bool[0] and keep_bool[-1] and np.any(~keep_bool):
        first_false = np.where(~keep_bool)[0][0]
        start = (first_false + 1) % n

        points = np.concatenate([points[start:], points[:start]], axis=0)
        keep_bool = np.concatenate([keep_bool[start:], keep_bool[:start]], axis=0)

    segments = []
    i = 0

    while i < n:
        if not keep_bool[i]:
            i += 1
            continue

        j = i
        while j < n and keep_bool[j]:
            j += 1

        seg = points[i:j]

        if len(seg) >= min_len:
            segments.append(seg)

        i = j

    return segments


def contorno_fondo_blanco_tejido_abierto(
    img_rgb,

    # --- detección de tejido morado/rosa ---
    tissue_sat_threshold=18,
    tissue_min_value=35,
    tissue_open_size=3,
    tissue_close_size=11,
    tissue_dilate_size=5,
    min_tissue_area=400,

    # --- detección de fondo blanco exterior ---
    white_sat_max=35,
    white_value_min=205,
    white_open_size=3,
    white_close_size=5,
    min_white_bg_area=300,

    # --- extracción de frontera ---
    adjacency_radius=5,
    min_segment_len=40,
    keep_only_longest=True,
    ignore_image_border_px=2,

    # --- dibujo ---
    line_color=(255, 255, 0),
    line_thickness=5,
    draw_dots=False,
    dot_spacing=8,
    dot_radius=3,

    show_steps=True
):
    """
    Extrae SOLO la frontera abierta entre fondo blanco exterior y espécimen.

    No busca contorno cerrado.
    No usa 'primer tejido desde arriba'.
    No coge huecos blancos internos si no están conectados al borde.
    No debería coger bordes negros ni laterales de imagen.

    img_rgb debe estar en RGB.
    """

    img = img_rgb.copy()
    h, w = img.shape[:2]

    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    H, S, V = cv2.split(hsv)

    # =========================================================
    # 1. Máscara inicial de tejido teñido
    # =========================================================
    tissue_core = (
        (S > tissue_sat_threshold) &
        (V > tissue_min_value)
    ).astype(np.uint8) * 255

    kernel_open = np.ones((tissue_open_size, tissue_open_size), np.uint8)
    tissue_core = cv2.morphologyEx(tissue_core, cv2.MORPH_OPEN, kernel_open)

    # Eliminar ruido pequeño de tejido
    nlab, labels, stats, _ = cv2.connectedComponentsWithStats(tissue_core, connectivity=8)

    tissue_clean = np.zeros_like(tissue_core)

    for lab in range(1, nlab):
        area = stats[lab, cv2.CC_STAT_AREA]
        if area >= min_tissue_area:
            tissue_clean[labels == lab] = 255

    # Unir ligeramente tejido fragmentado, pero sin cerrar grandes huecos externos
    kernel_dilate = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (tissue_dilate_size, tissue_dilate_size)
    )
    tissue_mask = cv2.dilate(tissue_clean, kernel_dilate, iterations=1)

    kernel_close = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (tissue_close_size, tissue_close_size)
    )
    tissue_mask = cv2.morphologyEx(tissue_mask, cv2.MORPH_CLOSE, kernel_close)

    tissue_mask = fill_holes_binary(tissue_mask)

    # =========================================================
    # 2. Máscara de fondo blanco
    #    Blanco = baja saturación + alto brillo
    # =========================================================
    white_mask = (
        (S < white_sat_max) &
        (V > white_value_min)
    ).astype(np.uint8) * 255

    kernel_white_open = np.ones((white_open_size, white_open_size), np.uint8)
    kernel_white_close = np.ones((white_close_size, white_close_size), np.uint8)

    white_mask = cv2.morphologyEx(white_mask, cv2.MORPH_OPEN, kernel_white_open)
    white_mask = cv2.morphologyEx(white_mask, cv2.MORPH_CLOSE, kernel_white_close)

    # Nos quedamos SOLO con fondo blanco conectado al borde.
    # Esto elimina huecos blancos internos del espécimen.
    external_white = keep_border_connected_components(
        white_mask,
        min_area=min_white_bg_area
    )

    if np.count_nonzero(external_white) == 0:
        raise ValueError(
            "No se ha detectado fondo blanco exterior conectado al borde. "
            "Prueba a bajar white_value_min o subir white_sat_max."
        )

    # =========================================================
    # 3. Frontera: puntos del contorno del fondo blanco que tocan tejido
    # =========================================================
    tissue_near = cv2.dilate(
        tissue_mask,
        cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE,
            (2 * adjacency_radius + 1, 2 * adjacency_radius + 1)
        ),
        iterations=1
    )

    contours_bg, _ = cv2.findContours(
        external_white,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_NONE
    )

    if len(contours_bg) == 0:
        raise ValueError("No se han encontrado contornos del fondo blanco exterior.")

    candidate_segments = []

    for cnt in contours_bg:
        pts = cnt.reshape(-1, 2)

        keep = []

        for x, y in pts:
            x = int(x)
            y = int(y)

            # evitar borde puro de imagen
            near_image_border = (
                x < ignore_image_border_px or
                y < ignore_image_border_px or
                x >= w - ignore_image_border_px or
                y >= h - ignore_image_border_px
            )

            if near_image_border:
                keep.append(False)
                continue

            # nos quedamos solo con puntos de fondo blanco cercanos a tejido
            keep.append(tissue_near[y, x] > 0)

        segments = get_open_runs_from_closed_contour(
            pts,
            keep,
            min_len=min_segment_len
        )

        candidate_segments.extend(segments)

    if len(candidate_segments) == 0:
        raise ValueError(
            "No se ha encontrado una frontera fondo blanco-tejido. "
            "Prueba a subir adjacency_radius o bajar min_segment_len."
        )

    # Elegir solo el segmento más largo, o todos los segmentos
    if keep_only_longest:
        lengths = [len(seg) for seg in candidate_segments]
        selected_segments = [candidate_segments[int(np.argmax(lengths))]]
    else:
        selected_segments = candidate_segments

    # =========================================================
    # 4. Dibujar resultado
    # =========================================================
    overlay = img.copy()

    for seg in selected_segments:
        pts_draw = seg.reshape(-1, 1, 2).astype(np.int32)

        if draw_dots:
            flat = pts_draw.reshape(-1, 2)
            for i in range(0, len(flat), dot_spacing):
                x, y = flat[i]
                cv2.circle(
                    overlay,
                    (int(x), int(y)),
                    dot_radius,
                    line_color,
                    thickness=-1
                )
        else:
            cv2.polylines(
                overlay,
                [pts_draw],
                isClosed=False,
                color=line_color,
                thickness=line_thickness
            )

    # =========================================================
    # 5. Visualización
    # =========================================================
    if show_steps:
        fig, axes = plt.subplots(1, 5, figsize=(24, 7))

        axes[0].imshow(img)
        axes[0].axis("off")
        axes[0].set_title("Imagen original")

        axes[1].imshow(tissue_mask, cmap="gray")
        axes[1].axis("off")
        axes[1].set_title("Máscara de espécimen")

        axes[2].imshow(external_white, cmap="gray")
        axes[2].axis("off")
        axes[2].set_title("Fondo blanco exterior")

        debug_boundary = img.copy()
        for seg in selected_segments:
            cv2.polylines(
                debug_boundary,
                [seg.reshape(-1, 1, 2).astype(np.int32)],
                isClosed=False,
                color=line_color,
                thickness=line_thickness
            )

        axes[3].imshow(debug_boundary)
        axes[3].axis("off")
        axes[3].set_title("Frontera seleccionada")

        axes[4].imshow(overlay)
        axes[4].axis("off")
        axes[4].set_title("Resultado final")

        plt.tight_layout()
        plt.show()

    return overlay, tissue_mask, external_white, selected_segments

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt


def fill_holes_binary(mask):
    """
    Rellena huecos internos de una máscara binaria.
    mask debe ser uint8 con valores 0/255.
    """
    mask = mask.copy()

    padded = cv2.copyMakeBorder(
        mask,
        1, 1, 1, 1,
        borderType=cv2.BORDER_CONSTANT,
        value=0
    )

    flood = padded.copy()
    flood_mask = np.zeros((padded.shape[0] + 2, padded.shape[1] + 2), np.uint8)

    cv2.floodFill(flood, flood_mask, (0, 0), 255)

    holes = cv2.bitwise_not(flood)
    filled = cv2.bitwise_or(padded, holes)

    return filled[1:-1, 1:-1]


def keep_border_connected_components(mask, min_area=500):
    """
    Se queda solo con componentes blancos conectados al borde de la imagen.
    Esto permite identificar fondo blanco exterior y descartar huecos internos.
    """
    h, w = mask.shape[:2]

    nlab, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)

    out = np.zeros_like(mask)

    for lab in range(1, nlab):
        x = stats[lab, cv2.CC_STAT_LEFT]
        y = stats[lab, cv2.CC_STAT_TOP]
        bw = stats[lab, cv2.CC_STAT_WIDTH]
        bh = stats[lab, cv2.CC_STAT_HEIGHT]
        area = stats[lab, cv2.CC_STAT_AREA]

        touches_border = (
            x == 0 or
            y == 0 or
            x + bw >= w or
            y + bh >= h
        )

        if touches_border and area >= min_area:
            out[labels == lab] = 255

    return out


def get_open_runs_from_closed_contour(points, keep_bool, min_len=30):
    """
    Dado un contorno cerrado y una máscara booleana de puntos válidos,
    devuelve segmentos abiertos consecutivos.
    """
    points = np.asarray(points)
    keep_bool = np.asarray(keep_bool).astype(bool)

    n = len(points)
    if n == 0 or not np.any(keep_bool):
        return []

    # Si el run cruza el inicio/final del contorno cerrado, rotamos
    if keep_bool[0] and keep_bool[-1] and np.any(~keep_bool):
        first_false = np.where(~keep_bool)[0][0]
        start = (first_false + 1) % n

        points = np.concatenate([points[start:], points[:start]], axis=0)
        keep_bool = np.concatenate([keep_bool[start:], keep_bool[:start]], axis=0)

    segments = []
    i = 0

    while i < n:
        if not keep_bool[i]:
            i += 1
            continue

        j = i
        while j < n and keep_bool[j]:
            j += 1

        seg = points[i:j]

        if len(seg) >= min_len:
            segments.append(seg)

        i = j

    return segments


def contorno_desde_mascara_especimen(
    img_rgb,

    # --- detección de tejido morado/rosa ---
    tissue_sat_threshold=18,
    tissue_min_value=35,
    tissue_open_size=3,
    tissue_close_size=11,
    tissue_dilate_size=5,
    min_tissue_area=400,

    # --- detección de fondo blanco exterior ---
    white_sat_max=35,
    white_value_min=205,
    white_open_size=3,
    white_close_size=5,
    min_white_bg_area=300,

    # --- extracción de frontera ---
    adjacency_radius=5,
    min_segment_len=40,
    keep_only_longest=True,
    ignore_image_border_px=2,

    # --- dibujo ---
    line_color=(255, 255, 0),
    line_thickness=6,
    draw_dots=False,
    dot_spacing=8,
    dot_radius=3,

    show_steps=True
):
    """
    Extrae el contorno a partir de la máscara del espécimen.

    Importante:
    - La forma de sacar la máscara del espécimen es la misma que en tu código.
    - El contorno se obtiene desde tissue_mask ya rellenada.
    - Luego se conserva solo la parte del contorno que toca fondo blanco exterior.
    - Así no se usan huecos internos ni manchas blancas internas.
    """

    img = img_rgb.copy()
    h, w = img.shape[:2]

    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    H, S, V = cv2.split(hsv)

    # =========================================================
    # 1. Máscara inicial de tejido teñido
    #    ESTA PARTE ESTÁ COPIADA DE TU CÓDIGO
    # =========================================================
    tissue_core = (
        (S > tissue_sat_threshold) &
        (V > tissue_min_value)
    ).astype(np.uint8) * 255

    kernel_open = np.ones((tissue_open_size, tissue_open_size), np.uint8)
    tissue_core = cv2.morphologyEx(tissue_core, cv2.MORPH_OPEN, kernel_open)

    # Eliminar ruido pequeño de tejido
    nlab, labels, stats, _ = cv2.connectedComponentsWithStats(tissue_core, connectivity=8)

    tissue_clean = np.zeros_like(tissue_core)

    for lab in range(1, nlab):
        area = stats[lab, cv2.CC_STAT_AREA]
        if area >= min_tissue_area:
            tissue_clean[labels == lab] = 255

    # Unir ligeramente tejido fragmentado, pero sin cerrar grandes huecos externos
    kernel_dilate = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (tissue_dilate_size, tissue_dilate_size)
    )
    tissue_mask = cv2.dilate(tissue_clean, kernel_dilate, iterations=1)

    kernel_close = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (tissue_close_size, tissue_close_size)
    )
    tissue_mask = cv2.morphologyEx(tissue_mask, cv2.MORPH_CLOSE, kernel_close)

    tissue_mask = fill_holes_binary(tissue_mask)

    # =========================================================
    # 2. Máscara de fondo blanco exterior
    #    Blanco = baja saturación + alto brillo
    # =========================================================
    white_mask = (
        (S < white_sat_max) &
        (V > white_value_min)
    ).astype(np.uint8) * 255

    kernel_white_open = np.ones((white_open_size, white_open_size), np.uint8)
    kernel_white_close = np.ones((white_close_size, white_close_size), np.uint8)

    white_mask = cv2.morphologyEx(white_mask, cv2.MORPH_OPEN, kernel_white_open)
    white_mask = cv2.morphologyEx(white_mask, cv2.MORPH_CLOSE, kernel_white_close)

    external_white = keep_border_connected_components(
        white_mask,
        min_area=min_white_bg_area
    )

    if np.count_nonzero(external_white) == 0:
        raise ValueError(
            "No se ha detectado fondo blanco exterior conectado al borde. "
            "Prueba a bajar white_value_min o subir white_sat_max."
        )

    # Dilatamos el fondo blanco exterior para comprobar contacto con el contorno del espécimen
    external_white_near = cv2.dilate(
        external_white,
        cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE,
            (2 * adjacency_radius + 1, 2 * adjacency_radius + 1)
        ),
        iterations=1
    )

    # =========================================================
    # 3. AHORA SÍ:
    #    Sacar contorno externo DESDE LA MÁSCARA DEL ESPÉCIMEN
    # =========================================================
    contours_specimen, _ = cv2.findContours(
        tissue_mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_NONE
    )

    if len(contours_specimen) == 0:
        raise ValueError("No se ha encontrado contorno externo del espécimen.")

    candidate_segments = []

    for cnt in contours_specimen:
        pts = cnt.reshape(-1, 2)

        keep = []

        for x, y in pts:
            x = int(x)
            y = int(y)

            # evitar borde puro de imagen
            near_image_border = (
                x < ignore_image_border_px or
                y < ignore_image_border_px or
                x >= w - ignore_image_border_px or
                y >= h - ignore_image_border_px
            )

            if near_image_border:
                keep.append(False)
                continue

            # nos quedamos solo con puntos del contorno del espécimen
            # que están cerca del fondo blanco exterior
            keep.append(external_white_near[y, x] > 0)

        segments = get_open_runs_from_closed_contour(
            pts,
            keep,
            min_len=min_segment_len
        )

        candidate_segments.extend(segments)

    if len(candidate_segments) == 0:
        raise ValueError(
            "No se ha encontrado ningún tramo del contorno del espécimen "
            "en contacto con fondo blanco exterior. "
            "Prueba a subir adjacency_radius o bajar min_segment_len."
        )

    # Elegir solo el segmento más largo, o todos los segmentos
    if keep_only_longest:
        lengths = [len(seg) for seg in candidate_segments]
        selected_segments = [candidate_segments[int(np.argmax(lengths))]]
    else:
        selected_segments = candidate_segments

    # =========================================================
    # 4. Dibujar resultado
    # =========================================================
    overlay = img.copy()

    for seg in selected_segments:
        pts_draw = seg.reshape(-1, 1, 2).astype(np.int32)

        if draw_dots:
            flat = pts_draw.reshape(-1, 2)
            for i in range(0, len(flat), dot_spacing):
                x, y = flat[i]
                cv2.circle(
                    overlay,
                    (int(x), int(y)),
                    dot_radius,
                    line_color,
                    thickness=-1
                )
        else:
            cv2.polylines(
                overlay,
                [pts_draw],
                isClosed=False,
                color=line_color,
                thickness=line_thickness
            )

    # =========================================================
    # 5. Visualización
    # =========================================================
    if show_steps:
        fig, axes = plt.subplots(1, 5, figsize=(24, 7))

        axes[0].imshow(img)
        axes[0].axis("off")
        axes[0].set_title("Imagen original")

        axes[1].imshow(tissue_mask, cmap="gray")
        axes[1].axis("off")
        axes[1].set_title("Máscara de espécimen rellena")

        axes[2].imshow(external_white, cmap="gray")
        axes[2].axis("off")
        axes[2].set_title("Fondo blanco exterior")

        debug_all_contours = img.copy()
        cv2.drawContours(
            debug_all_contours,
            contours_specimen,
            -1,
            line_color,
            thickness=line_thickness
        )

        axes[3].imshow(debug_all_contours)
        axes[3].axis("off")
        axes[3].set_title("Contorno externo de la máscara")

        axes[4].imshow(overlay)
        axes[4].axis("off")
        axes[4].set_title("Tramo final seleccionado")

        plt.tight_layout()
        plt.show()

    return overlay, tissue_mask, external_white, selected_segments

In [ ]:
img_bgr = cv2.imread(r"Datos\SB013\SB013\H&E_python\tiles_level4_seleccionadas\tile_051_r04_c06_x4800_y5600_w800_h1400.png")
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

overlay_dots, overlay_line, mask, xs, ys = contorno_abierto_superior_he(
    img_rgb,
    sat_threshold=20,
    min_value=35,
    kernel_close_size=15,
    min_component_area=500,
    smooth_window=21,
    dot_spacing=6,
    dot_radius=3,
    show_steps=True
)

In [ ]:
img_bgr = cv2.imread(r"Datos\SB013\SB013\H&E_python\tiles_level4_seleccionadas\tile_051_r04_c06_x4800_y5600_w800_h1400.png")
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

overlay, tissue_mask, external_white, segments = contorno_fondo_blanco_tejido_abierto(
    img_rgb,
    tissue_sat_threshold=18,
    white_value_min=205,
    adjacency_radius=5,
    min_segment_len=40,
    line_thickness=5,
    keep_only_longest=True,
    show_steps=True
)

In [ ]:
img_bgr = cv2.imread(r"Datos\SB013\SB013\H&E_python\tiles_level4_seleccionadas\tile_051_r04_c06_x4800_y5600_w800_h1400.png")
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

overlay, tissue_mask, external_white, segments = contorno_desde_mascara_especimen(
    img_rgb,
    tissue_sat_threshold=18,
    tissue_min_value=35,
    tissue_open_size=3,
    tissue_close_size=11,
    tissue_dilate_size=5,
    min_tissue_area=400,
    white_sat_max=35,
    white_value_min=205,
    adjacency_radius=5,
    min_segment_len=40,
    keep_only_longest=True,
    line_thickness=6,
    show_steps=True
)

In [ ]:
img_bgr = cv2.imread(r"Datos\SB013\SB013\H&E_python\tiles_level4_seleccionadas\tile_091_r08_c02_x1600_y11200_w800_h1400.png")
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

overlay_dots, overlay_line, mask, xs, ys = contorno_abierto_superior_he(
    img_rgb,
    sat_threshold=20,
    min_value=35,
    kernel_close_size=15,
    min_component_area=500,
    smooth_window=21,
    dot_spacing=6,
    dot_radius=3,
    show_steps=True
)

In [ ]:
img_bgr = cv2.imread(r"Datos\SB013\SB013\H&E_python\tiles_level4_seleccionadas\tile_035_r03_c01_x800_y4200_w800_h1400.png")
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

overlay_dots, overlay_line, mask, xs, ys = contorno_abierto_superior_he(
    img_rgb,
    sat_threshold=20,
    min_value=35,
    kernel_close_size=15,
    min_component_area=500,
    smooth_window=21,
    dot_spacing=6,
    dot_radius=3,
    show_steps=True
)

In [ ]:
img_bgr = cv2.imread(r"Datos\SB013\SB013\H&E_python\tiles_level4_seleccionadas\tile_104_r09_c04_x3200_y12600_w800_h1400.png")
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

overlay_dots, overlay_line, mask, xs, ys = contorno_abierto_superior_he(
    img_rgb,
    sat_threshold=20,
    min_value=35,
    kernel_close_size=15,
    min_component_area=500,
    smooth_window=21,
    dot_spacing=6,
    dot_radius=3,
    show_steps=True
)

In [ ]:
# ============================================================
# 7. Ejemplo de uso
# ============================================================

tile_path = r"Datos\SB013\SB013\H&E_python\tiles_level4_seleccionadas\tile_104_r09_c04_x3200_y12600_w800_h1400.png"

img, result = process_tile_debug(
    tile_path,

    show_debug=True,
    show_final=True,
    save_outputs=False,

    # Modo agresivo
    white_v_threshold=190,
    white_s_threshold=65,
    white_dist_threshold=105,

    black_v_threshold=35,
    black_mean_threshold=45,

    seed_border_width=4,
    candidate_close_size=5,

    min_bg_area_frac=0.005,
    min_bg_area_px=2500,
    min_bg_border_touch_frac=0.005,
    min_bg_thickness_px=8,

    contour_thickness=2,
)

Más prubeas

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


# ============================================================
# 1. Lectura y utilidades básicas
# ============================================================

def read_rgb(path):
    """
    Lee una imagen con OpenCV y la devuelve en RGB.
    """
    img_bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)

    if img_bgr is None:
        raise FileNotFoundError(f"No se pudo leer la imagen: {path}")

    return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)


def ensure_uint8_rgb(img):
    """
    Asegura formato RGB uint8.
    """
    img = np.asarray(img)

    if img.ndim != 3 or img.shape[2] != 3:
        raise ValueError("La imagen debe tener forma H x W x 3")

    if img.dtype != np.uint8:
        img = np.clip(img, 0, 255).astype(np.uint8)

    return img


def border_mask(shape, width=3):
    """
    Crea una máscara booleana del borde de la imagen.
    """
    h, w = shape

    mask = np.zeros((h, w), dtype=bool)

    mask[:width, :] = True
    mask[-width:, :] = True
    mask[:, :width] = True
    mask[:, -width:] = True

    return mask


def overlay_mask(img_rgb, mask, color=(255, 0, 0), alpha=0.6):
    """
    Pinta una máscara sobre la imagen.

    color en RGB:
    rojo     = (255, 0, 0)
    verde    = (0, 255, 0)
    amarillo = (255, 255, 0)
    """
    img_rgb = ensure_uint8_rgb(img_rgb)

    out = img_rgb.copy().astype(np.float32)
    mask = mask.astype(bool)

    color = np.array(color, dtype=np.float32)

    out[mask] = (1 - alpha) * out[mask] + alpha * color

    return np.clip(out, 0, 255).astype(np.uint8)


def overlay_contour(img_rgb, contour_mask, color=(255, 0, 0)):
    """
    Pinta el contorno sobre la imagen.
    """
    img_rgb = ensure_uint8_rgb(img_rgb)

    out = img_rgb.copy()
    contour_mask = contour_mask.astype(bool)

    out[contour_mask] = np.array(color, dtype=np.uint8)

    return out


# ============================================================
# 2. Detectar negro del padding/corte escaneado
# ============================================================

def detect_black_padding(
    img_rgb,
    black_v_threshold=35,
    black_mean_threshold=45,
    keep_only_border_connected=True,
    border_width=3,
):
    """
    Detecta el negro del padding/corte escaneado.

    Si keep_only_border_connected=True:
    solo se queda con componentes negras conectadas al borde.
    Esto evita coger puntos negros internos del tejido.
    """

    img_rgb = ensure_uint8_rgb(img_rgb)

    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    _, _, V = cv2.split(hsv)

    mean_rgb = img_rgb.mean(axis=2)

    black_mask_raw = (
        (V <= black_v_threshold) &
        (mean_rgb <= black_mean_threshold)
    )

    if not keep_only_border_connected:
        return black_mask_raw

    h, w = black_mask_raw.shape
    bmask = border_mask((h, w), width=border_width)

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        black_mask_raw.astype(np.uint8),
        connectivity=8
    )

    border_labels = np.unique(labels[black_mask_raw & bmask])
    border_labels = border_labels[border_labels != 0]

    black_padding_mask = np.isin(labels, border_labels)

    return black_padding_mask


# ============================================================
# 3. Detectar píxeles blancos candidatos a background
# ============================================================

def detect_white_background_candidates(
    img_rgb,
    white_v_threshold=190,
    white_s_threshold=50,
    white_dist_threshold=90,
):
    """
    Detecta píxeles blancos/casi blancos candidatos a background.

    Ojo:
    aquí entrarán también blancos internos del espécimen.
    Por eso luego se selecciona el cluster blanco más cercano al negro.
    """

    img_rgb = ensure_uint8_rgb(img_rgb)

    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    _, S, V = cv2.split(hsv)

    img_i16 = img_rgb.astype(np.int16)

    dist_to_white = np.sqrt(
        np.sum((255 - img_i16) ** 2, axis=2)
    )

    white_candidates = (
        (V >= white_v_threshold) &
        (S <= white_s_threshold) &
        (dist_to_white <= white_dist_threshold)
    )

    return white_candidates


# ============================================================
# 4. Elegir el cluster blanco más cercano al negro
# ============================================================

def select_white_cluster_nearest_to_black(
    white_candidates,
    black_padding_mask,
    min_white_cluster_area_px=8000,
):
    """
    De todos los clusters blancos candidatos, se queda con el cluster
    cuya distancia mínima al background negro sea menor.

    Devuelve:
    - selected_white_bg_mask
    - labels
    - selected_info
    - all_components_info
    """

    white_candidates = white_candidates.astype(bool)
    black_padding_mask = black_padding_mask.astype(bool)

    h, w = white_candidates.shape

    selected_white_bg_mask = np.zeros((h, w), dtype=bool)

    # Si no hay negro, no seleccionamos background final.
    if black_padding_mask.sum() == 0:
        return selected_white_bg_mask, None, None, []

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        white_candidates.astype(np.uint8),
        connectivity=8
    )

    # Distancia al negro
    non_black = (~black_padding_mask).astype(np.uint8)
    dist_to_black = cv2.distanceTransform(non_black, cv2.DIST_L2, 5)

    all_components_info = []

    for lab in range(1, num_labels):
        area = int(stats[lab, cv2.CC_STAT_AREA])

        if area < min_white_cluster_area_px:
            continue

        component = labels == lab

        distances = dist_to_black[component]

        min_dist = float(distances.min())
        mean_dist = float(distances.mean())

        info = {
            "label": int(lab),
            "area": area,
            "min_dist_to_black": min_dist,
            "mean_dist_to_black": mean_dist,
        }

        all_components_info.append(info)

    if len(all_components_info) == 0:
        return selected_white_bg_mask, labels, None, []

    # Orden:
    # 1) menor distancia mínima al negro
    # 2) si empatan, cluster más grande
    all_components_info = sorted(
        all_components_info,
        key=lambda x: (x["min_dist_to_black"], -x["area"])
    )

    selected_info = all_components_info[0]
    selected_label = selected_info["label"]

    selected_white_bg_mask = labels == selected_label

    return selected_white_bg_mask, labels, selected_info, all_components_info


# ============================================================
# 5. Utilidades para "forzar línea"
# ============================================================

def morphological_skeleton(binary_mask):
    """
    Skeleton morfológico de una máscara binaria.
    Devuelve una línea de 1 píxel aprox.
    """
    img = (binary_mask.astype(np.uint8) * 255).copy()
    skel = np.zeros_like(img)

    kernel = cv2.getStructuringElement(cv2.MORPH_CROSS, (3, 3))

    while True:
        eroded = cv2.erode(img, kernel)
        temp = cv2.dilate(eroded, kernel)
        temp = cv2.subtract(img, temp)
        skel = cv2.bitwise_or(skel, temp)
        img = eroded.copy()

        if cv2.countNonZero(img) == 0:
            break

    return skel > 0


def count_endpoints(component_mask):
    """
    Cuenta endpoints de una línea esquelética.
    Un endpoint es un píxel del skeleton con exactamente 1 vecino
    en conectividad-8.
    """
    comp = component_mask.astype(np.uint8)

    kernel = np.array([
        [1, 1, 1],
        [1, 10, 1],
        [1, 1, 1]
    ], dtype=np.uint8)

    conv = cv2.filter2D(comp, -1, kernel, borderType=cv2.BORDER_CONSTANT)

    # Si el centro está activo (10) y tiene un solo vecino -> 11
    endpoints = (comp == 1) & (conv == 11)

    return int(endpoints.sum())


def keep_only_open_line_components(skeleton_mask, min_component_length=15):
    """
    Mantiene solo componentes del skeleton que sean líneas abiertas.
    Elimina:
    - círculos cerrados (0 endpoints)
    - componentes demasiado pequeñas
    """
    skeleton_mask = skeleton_mask.astype(bool)

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        skeleton_mask.astype(np.uint8),
        connectivity=8
    )

    out = np.zeros_like(skeleton_mask, dtype=bool)

    for lab in range(1, num_labels):
        component = labels == lab
        area = int(stats[lab, cv2.CC_STAT_AREA])

        if area < min_component_length:
            continue

        n_endpoints = count_endpoints(component)

        # Nos quedamos con líneas abiertas:
        # normalmente 2 endpoints.
        # Permitimos >=1 por si hay algún caso degenerado.
        # Un círculo cerrado tendrá 0 endpoints -> se elimina.
        if n_endpoints >= 1:
            out |= component

    return out


# ============================================================
# 6. Sacar contorno desde el background final
# ============================================================
def detect_uniform_black_side_strip(
    black_padding_mask,
    min_coverage_frac=0.95,
    max_thickness_std=1.5,
    max_thickness_range=4,
    min_mean_thickness=1,
    max_mean_thickness=15,   # NUEVO: grosor medio máximo permitido
):
    """
    Detecta si existe una franja negra uniforme y fina ocupando uno de los 4 lados
    de la imagen (top, bottom, left, right).

    Condición:
    - la franja debe recorrer prácticamente todo el lado
    - su grosor debe ser aproximadamente constante
    - su grosor medio debe estar entre min_mean_thickness y max_mean_thickness

    Devuelve:
    - found: bool
    - side: 'top' | 'bottom' | 'left' | 'right' | None
    - thicknesses: vector de espesores por columna/fila
    """

    black = black_padding_mask.astype(bool)
    h, w = black.shape

    def evaluate_side(thicknesses):
        present = thicknesses > 0
        coverage_frac = present.mean()

        if coverage_frac < min_coverage_frac:
            return False

        t = thicknesses[present]
        if len(t) == 0:
            return False

        mean_t = float(np.mean(t))
        std_t = float(np.std(t))
        range_t = int(np.max(t) - np.min(t))

        # Grosor mínimo
        if mean_t < min_mean_thickness:
            return False

        # NUEVO: grosor máximo
        if mean_t > max_mean_thickness:
            return False

        # Uniformidad por desviación estándar
        if std_t > max_thickness_std:
            return False

        # Uniformidad por rango máximo-mínimo
        if range_t > max_thickness_range:
            return False

        return True

    # --------------------------------------------------------
    # TOP: grosor negro desde arriba hacia abajo, por columna
    # --------------------------------------------------------
    top_thickness = np.zeros(w, dtype=int)
    for x in range(w):
        t = 0
        while t < h and black[t, x]:
            t += 1
        top_thickness[x] = t

    if evaluate_side(top_thickness):
        return True, "top", top_thickness

    # --------------------------------------------------------
    # BOTTOM: grosor negro desde abajo hacia arriba, por columna
    # --------------------------------------------------------
    bottom_thickness = np.zeros(w, dtype=int)
    for x in range(w):
        t = 0
        y = h - 1
        while y - t >= 0 and black[y - t, x]:
            t += 1
        bottom_thickness[x] = t

    if evaluate_side(bottom_thickness):
        return True, "bottom", bottom_thickness

    # --------------------------------------------------------
    # LEFT: grosor negro desde izquierda hacia derecha, por fila
    # --------------------------------------------------------
    left_thickness = np.zeros(h, dtype=int)
    for y in range(h):
        t = 0
        while t < w and black[y, t]:
            t += 1
        left_thickness[y] = t

    if evaluate_side(left_thickness):
        return True, "left", left_thickness

    # --------------------------------------------------------
    # RIGHT: grosor negro desde derecha hacia izquierda, por fila
    # --------------------------------------------------------
    right_thickness = np.zeros(h, dtype=int)
    for y in range(h):
        t = 0
        x = w - 1
        while x - t >= 0 and black[y, x - t]:
            t += 1
        right_thickness[y] = t

    if evaluate_side(right_thickness):
        return True, "right", right_thickness

    return False, None, None


def extract_contour_from_final_background(
    selected_white_bg_mask,
    black_padding_mask,
    contour_thickness=2,
    force_open_line=True,
    min_component_length=15,

    # compatibilidad con tu llamada actual
    img_rgb=None,
    white_candidates=None,
    use_black_touching_specimen_fallback=True,
    min_black_touching_line_length=30,

    # parámetros del nuevo caso
    use_uniform_side_black_strip_rule=True,
    min_side_strip_coverage_frac=0.95,
    max_side_strip_thickness_std=1.5,
    max_side_strip_thickness_range=4,
):
    """
    Saca el contorno entre background y espécimen.

    NUEVA REGLA PRIORITARIA:
    Si existe una franja negra uniforme ocupando un lado completo
    de la imagen, el contorno se fuerza como la línea interior
    de esa franja negra.

    Si no, sigue con el caso normal:
    contorno entre background blanco final y espécimen.
    """

    bg_mask = selected_white_bg_mask.astype(bool)
    black_mask = black_padding_mask.astype(bool)

    if black_mask.sum() == 0:
        contour_mask = np.zeros_like(bg_mask, dtype=bool)
        return contour_mask, False, "no se encontró contorno"

    h, w = black_mask.shape

    # ========================================================
    # NUEVA REGLA:
    # franja negra uniforme en uno de los lados
    # ========================================================
    if use_uniform_side_black_strip_rule:
        found_strip, strip_side, strip_thicknesses = detect_uniform_black_side_strip(
            black_mask,
            min_coverage_frac=min_side_strip_coverage_frac,
            max_thickness_std=max_side_strip_thickness_std,
            max_thickness_range=max_side_strip_thickness_range,
            min_mean_thickness=1,
            max_mean_thickness=15,   # NUEVO
        )

        if found_strip:
            contour_line = np.zeros_like(black_mask, dtype=bool)

            if strip_side == "top":
                for x in range(w):
                    t = strip_thicknesses[x]
                    if t > 0 and t < h:
                        contour_line[t, x] = True

            elif strip_side == "bottom":
                for x in range(w):
                    t = strip_thicknesses[x]
                    if t > 0 and t < h:
                        y = h - t - 1
                        if 0 <= y < h:
                            contour_line[y, x] = True

            elif strip_side == "left":
                for y in range(h):
                    t = strip_thicknesses[y]
                    if t > 0 and t < w:
                        contour_line[y, t] = True

            elif strip_side == "right":
                for y in range(h):
                    t = strip_thicknesses[y]
                    if t > 0 and t < w:
                        x = w - t - 1
                        if 0 <= x < w:
                            contour_line[y, x] = True

            # adelgazar / limpiar si quieres mantener el mismo comportamiento
            if contour_line.sum() > 0 and force_open_line:
                contour_line = morphological_skeleton(contour_line)

                contour_line = keep_only_open_line_components(
                    contour_line,
                    min_component_length=min_component_length
                )

            if contour_line.sum() > 0:
                return contour_line, True, f"contorno detectado usando franja negra uniforme en {strip_side}"

    # ========================================================
    # CASO NORMAL:
    # contorno entre background blanco final y espécimen
    # ========================================================
    if bg_mask.sum() == 0:
        contour_mask = np.zeros_like(bg_mask, dtype=bool)
        return contour_mask, False, "no se encontró contorno"

    specimen_mask = ~(bg_mask | black_mask)

    ksize = max(3, 2 * contour_thickness + 1)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (ksize, ksize))

    bg_dilated = cv2.dilate(bg_mask.astype(np.uint8), kernel) > 0

    # Solo píxeles del espécimen tocando background blanco
    contour_mask = bg_dilated & specimen_mask

    if contour_mask.sum() == 0:
        contour_mask = np.zeros_like(bg_mask, dtype=bool)
        return contour_mask, False, "no se encontró contorno"

    if force_open_line:
        contour_skeleton = morphological_skeleton(contour_mask)

        contour_line = keep_only_open_line_components(
            contour_skeleton,
            min_component_length=min_component_length
        )

        if contour_line.sum() == 0:
            contour_line = np.zeros_like(bg_mask, dtype=bool)
            return contour_line, False, "no se encontró contorno"

        return contour_line, True, "contorno detectado"

    return contour_mask, True, "contorno detectado"


# ============================================================
# 7. Pipeline completo
# ============================================================

def pipeline_background_from_black_then_white(
    img_rgb,

    # Negro
    black_v_threshold=35,
    black_mean_threshold=45,
    black_border_width=3,

    # Blanco candidato
    white_v_threshold=190,
    white_s_threshold=50,
    white_dist_threshold=90,

    # Cluster blanco
    min_white_cluster_area_px=8000,

    # Contorno
    contour_thickness=2,
    force_open_line=True,
    min_component_length=15,
):
    """
    Pipeline:

    1. Detectar negro del padding/corte.
    2. Pintar negro en rojo.
    3. Detectar blancos candidatos.
    4. Pintar blancos candidatos en verde.
    5. Seleccionar cluster blanco más cercano al negro.
    6. Pintar background final en amarillo.
    7. Sacar contorno entre background final y espécimen.
    """

    img_rgb = ensure_uint8_rgb(img_rgb)

    # Negro del padding/corte
    black_padding_mask = detect_black_padding(
        img_rgb,
        black_v_threshold=black_v_threshold,
        black_mean_threshold=black_mean_threshold,
        keep_only_border_connected=True,
        border_width=black_border_width,
    )

    # Blancos candidatos
    white_candidates = detect_white_background_candidates(
        img_rgb,
        white_v_threshold=white_v_threshold,
        white_s_threshold=white_s_threshold,
        white_dist_threshold=white_dist_threshold,
    )

    # Cluster blanco final
    selected_white_bg_mask, white_labels, selected_info, all_components_info = (
        select_white_cluster_nearest_to_black(
            white_candidates=white_candidates,
            black_padding_mask=black_padding_mask,
            min_white_cluster_area_px=min_white_cluster_area_px,
        )
    )

    # Contorno
    contour_mask, has_contour, contour_message = extract_contour_from_final_background(
        selected_white_bg_mask=selected_white_bg_mask,
        black_padding_mask=black_padding_mask,
        contour_thickness=contour_thickness,
        force_open_line=force_open_line,
        min_component_length=min_component_length,
    )

    result = {
        "black_padding_mask": black_padding_mask,
        "white_candidates": white_candidates,
        "selected_white_bg_mask": selected_white_bg_mask,
        "white_labels": white_labels,
        "selected_info": selected_info,
        "all_components_info": all_components_info,

        "contour_mask": contour_mask,
        "has_contour": has_contour,
        "contour_message": contour_message,
    }

    return result


# ============================================================
# 8. Visualización de una sola imagen
# ============================================================

def show_pipeline_background_debug(img_rgb, result, figsize=(28, 7)):
    """
    Muestra una imagen con:
    1. original
    2. negro en rojo
    3. blancos candidatos en verde
    4. cluster blanco final en amarillo
    5. contorno final
    """

    img_rgb = ensure_uint8_rgb(img_rgb)

    black_overlay = overlay_mask(
        img_rgb,
        result["black_padding_mask"],
        color=(255, 0, 0),
        alpha=0.75
    )

    white_overlay = overlay_mask(
        img_rgb,
        result["white_candidates"],
        color=(0, 255, 0),
        alpha=0.55
    )

    final_overlay = overlay_mask(
        img_rgb,
        result["selected_white_bg_mask"],
        color=(255, 255, 0),
        alpha=0.70
    )

    fig, axes = plt.subplots(1, 5, figsize=figsize)

    axes[0].imshow(img_rgb)
    axes[0].set_title("1. Imagen original")
    axes[0].axis("off")

    axes[1].imshow(black_overlay)
    axes[1].set_title("2-3. Negro padding/corte\nen rojo")
    axes[1].axis("off")

    axes[2].imshow(white_overlay)
    axes[2].set_title("4-5. Blancos candidatos\nen verde")
    axes[2].axis("off")

    axes[3].imshow(final_overlay)
    axes[3].set_title("6-7. Cluster blanco final\nen amarillo")
    axes[3].axis("off")

    if result["has_contour"]:
        contour_overlay = overlay_contour(
            img_rgb,
            result["contour_mask"],
            color=(255, 0, 0)
        )
        axes[4].imshow(contour_overlay)
        axes[4].set_title("Contorno detectado\nen rojo")
        axes[4].axis("off")
    else:
        axes[4].imshow(img_rgb)
        axes[4].set_title("Sin contorno")
        axes[4].axis("off")
        axes[4].text(
            0.5, 0.5,
            "no se encontró contorno",
            color="red",
            fontsize=14,
            ha="center",
            va="center",
            transform=axes[4].transAxes,
            bbox=dict(facecolor="white", alpha=0.8, edgecolor="red")
        )

    plt.tight_layout()
    plt.show()


# ============================================================
# 9. Visualización de varias imágenes
# ============================================================

def show_pipeline_background_debug_multiple(
    imgs,
    results,
    tile_paths=None,
    figsize=(30, 18)
):
    """
    Muestra varias imágenes a la vez.

    Cada fila = una imagen.

    Columnas:
    1. original
    2. negro en rojo
    3. blancos candidatos en verde
    4. cluster blanco final en amarillo
    5. contorno final
    """

    n = len(imgs)

    fig, axes = plt.subplots(n, 5, figsize=figsize)

    if n == 1:
        axes = np.expand_dims(axes, axis=0)

    for i, (img_rgb, result) in enumerate(zip(imgs, results)):

        img_rgb = ensure_uint8_rgb(img_rgb)

        black_overlay = overlay_mask(
            img_rgb,
            result["black_padding_mask"],
            color=(255, 0, 0),
            alpha=0.75
        )

        white_overlay = overlay_mask(
            img_rgb,
            result["white_candidates"],
            color=(0, 255, 0),
            alpha=0.55
        )

        final_overlay = overlay_mask(
            img_rgb,
            result["selected_white_bg_mask"],
            color=(255, 255, 0),
            alpha=0.70
        )

        if tile_paths is not None:
            row_title = Path(tile_paths[i]).name
        else:
            row_title = f"Imagen {i + 1}"

        axes[i, 0].imshow(img_rgb)
        axes[i, 0].set_title(f"{row_title}\nOriginal")
        axes[i, 0].axis("off")

        axes[i, 1].imshow(black_overlay)
        axes[i, 1].set_title("Negro padding/corte\nen rojo")
        axes[i, 1].axis("off")

        axes[i, 2].imshow(white_overlay)
        axes[i, 2].set_title("Blancos candidatos\nen verde")
        axes[i, 2].axis("off")

        axes[i, 3].imshow(final_overlay)
        axes[i, 3].set_title("Cluster blanco final\nen amarillo")
        axes[i, 3].axis("off")

        if result["has_contour"]:
            contour_overlay = overlay_contour(
                img_rgb,
                result["contour_mask"],
                color=(255, 0, 0)
            )
            axes[i, 4].imshow(contour_overlay)
            axes[i, 4].set_title("Contorno en rojo")
            axes[i, 4].axis("off")
        else:
            axes[i, 4].imshow(img_rgb)
            axes[i, 4].set_title("Sin contorno")
            axes[i, 4].axis("off")
            axes[i, 4].text(
                0.5, 0.5,
                "no se encontró contorno",
                color="red",
                fontsize=12,
                ha="center",
                va="center",
                transform=axes[i, 4].transAxes,
                bbox=dict(facecolor="white", alpha=0.8, edgecolor="red")
            )

    plt.tight_layout()
    plt.show()


# ============================================================
# 10. Resumen por pantalla
# ============================================================

def print_pipeline_summary(result):
    """
    Imprime resumen del pipeline.
    """

    print("====================================")
    print("BLACK PIXELS:", int(result["black_padding_mask"].sum()))
    print("WHITE CANDIDATE PIXELS:", int(result["white_candidates"].sum()))
    print("FINAL WHITE BACKGROUND PIXELS:", int(result["selected_white_bg_mask"].sum()))
    print("HAS_CONTOUR:", result["has_contour"])
    print("CONTOUR MESSAGE:", result["contour_message"])
    print("====================================")

    print("\nCLUSTER SELECCIONADO:")
    if result["selected_info"] is None:
        print("No se seleccionó ningún cluster blanco.")
    else:
        print(result["selected_info"])

    print("\nTODOS LOS CLUSTERS BLANCOS CONSIDERADOS:")
    if len(result["all_components_info"]) == 0:
        print("Ninguno")
    else:
        for c in result["all_components_info"]:
            print(c)

    print("====================================")


# ============================================================
# 11. Procesar todas las imágenes .png de una carpeta
# ============================================================

folder_path = Path(
    r"Datos\SB013\SB013\H&E_python\tiles_level4_seleccionadas"
)

# Coger todos los PNG de esa carpeta, ordenados por nombre
tile_paths = sorted([
    p for p in folder_path.iterdir()
    if p.is_file() and p.suffix.lower() == ".png"
])

print(f"Número de imágenes .png encontradas: {len(tile_paths)}")

if len(tile_paths) == 0:
    raise FileNotFoundError(f"No se encontraron imágenes .png en: {folder_path}")


# Procesar todas las imágenes
imgs = []
results = []

for tile_path in tile_paths:
    img = read_rgb(tile_path)

    result = pipeline_background_from_black_then_white(
        img,

        # Negro del corte/padding
        black_v_threshold=35,
        black_mean_threshold=45,
        black_border_width=3,

        # Blanco candidato
        white_v_threshold=190,
        white_s_threshold=50,
        white_dist_threshold=90,

        # Cluster blanco final
        min_white_cluster_area_px=8000,

        # Contorno
        contour_thickness=2,
        force_open_line=True,
        min_component_length=15,
    )

    imgs.append(img)
    results.append(result)

    print("\n\n==============================================")
    print("TILE:", tile_path.name)
    print_pipeline_summary(result)
    
    
# ============================================================
# 12. Visualizar resultados por bloques
# ============================================================

batch_size = 3

for start in range(0, len(tile_paths), batch_size):
    end = start + batch_size

    imgs_batch = imgs[start:end]
    results_batch = results[start:end]
    paths_batch = tile_paths[start:end]

    print(f"\nMostrando imágenes {start + 1} a {min(end, len(tile_paths))} de {len(tile_paths)}")

    show_pipeline_background_debug_multiple(
        imgs_batch,
        results_batch,
        tile_paths=paths_batch,
        figsize=(30, 6 * len(imgs_batch))
    )